# 03 — Unitree VLA (UnifoLM-VLA) evaluation harness

Scaffold for loading Unitree VLA checkpoints / predictions and comparing against the S2R stack
(Qwen reasoner + your VLA + ESN).

Official references:
- https://github.com/unitreerobotics/unifolm-vla
- Unitree HF datasets listed in `data/benchmark/tasks.yaml`

In [ ]:
from pathlib import Path
import sys, json
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

from s2r.experiments.paths import UNITREE_VLA, ensure_experiment_dirs
from s2r.experiments.benchmark import load_tasks, score_episode, save_score, summarize_results
from s2r.models.registry import build_reasoner, build_detector
from s2r.core.config import load_config
from s2r.models.yolo_detector import encode_image_stub

ensure_experiment_dirs()
cfg = load_config(ROOT / "config" / "platforms" / "g1_edu.yaml")
tasks = load_tasks()
print("unitree_vla dir", UNITREE_VLA)
print("checkpoints", list((UNITREE_VLA / "checkpoints").glob("*"))[:10])

In [ ]:
# Placeholder adapter — replace with UnifoLM-VLA inference API when installed.
class UnitreeVLAAdapter:
    def __init__(self, checkpoint: str | None = None):
        self.checkpoint = checkpoint
        self.name = "unitree_unifolm_vla"

    def act(self, instruction: str, image=None):
        # Return a sparse action token-like vector (dim=7) for pairing with ESN later
        import numpy as np
        rng = np.random.default_rng(abs(hash(instruction)) % (2**32))
        return {
            "action": rng.normal(scale=0.2, size=7).tolist(),
            "confidence": 0.8,
            "goal": instruction,
            "backend": self.name,
            "checkpoint": self.checkpoint,
        }

vla = UnitreeVLAAdapter(checkpoint=str(UNITREE_VLA / "checkpoints" / "unifolm_vla0.pt"))
sample = vla.act(tasks[0].instruction)
sample

In [ ]:
# Optional: run S2R perception/reasoner on the same instruction for side-by-side debug
det = build_detector(cfg)
reasoner = build_reasoner(cfg)
frame = det.infer(encode_image_stub(), prompt=tasks[4].instruction)  # pack pencilbox
plan = reasoner.plan(tasks[4].instruction, frame, {"holding_pen": False}, mission_phase="locate")
{
    "task": tasks[4].id,
    "perception": frame.caption,
    "objects": frame.objects_of_interest,
    "r2s_intent": plan.intent,
    "unitree_vla_action": vla.act(tasks[4].instruction),
}

In [ ]:
# Save prediction artifact for a task
task = tasks[4]
pred = {
    "task_id": task.id,
    "instruction": task.instruction,
    "model": vla.name,
    "action_token": vla.act(task.instruction),
}
out = UNITREE_VLA / "predictions" / f"{task.id}__demo.json"
out.write_text(json.dumps(pred, indent=2))
print("wrote", out)

# Example eval score row
save_score(score_episode(
    task_id=task.id,
    model_name=vla.name,
    episode_id="notebook_demo",
    success=True,
    completion_time_s=38.0,
    vla_hz=2.0,
    esn_hz=0.0,
    e2e_latency_ms=120.0,
))
pd_rows = summarize_results()
[r for r in pd_rows if r["model"] == vla.name][:3]

## Next steps on hardware

1. Put UnifoLM-VLA weights in `data/unitree_vla/checkpoints/`
2. Replace `UnitreeVLAAdapter.act` with real forward pass
3. Optionally feed Unitree VLA tokens into S2R `ESNUpsampleNode` via ZMQ `action_token` topic
4. Log episodes into `data/benchmark/tasks/<id>/episodes/`
5. Build leaderboard in notebook `02_benchmark_12_tasks.ipynb`